# Verify Binarized CSV File

Steps:
1. Verify cell barcode alignment between CSV and MuData
2. Drop isotype controls
3. Create new MuData with binarized proteins
4. Save new `.h5mu` file

In [7]:
import numpy as np
import pandas as pd
import muon as mu
from scipy.sparse import issparse, csr_matrix
import anndata as ad

DATA_PATH = '/Users/teosecilmis/Documents/CITESEQ/Teo_datasets/Hackathon files/'
CSV_PATH = '/Users/teosecilmis/Documents/Citeseq-Hackathon/Binarized_File/binarized.csv'

## Step 1 — Load and verify cell barcode alignment

In [8]:
# Load binarized CSV
binarized = pd.read_csv(CSV_PATH, index_col=0)

In [9]:
# Load MuData
mdata = mu.read(DATA_PATH + 'GSE194315_raw_mu.h5mu')

/opt/anaconda3/envs/teo_citeprep_env/lib/python3.11/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/opt/anaconda3/envs/teo_citeprep_env/lib/python3.11/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


## Step 2 — Drop isotype controls

In [10]:
# Identify isotype control columns
isotype_cols = [c for c in binarized.columns if 'isotype' in c.lower() or 'isotypeCtrl' in c]
print(f'Isotype control columns ({len(isotype_cols)}):')
for c in isotype_cols:
    print(f'  {c}')

# Drop them
binarized_clean = binarized.drop(columns=isotype_cols)
print(f'\nAfter dropping isotype controls: {binarized_clean.shape}')
print(f'Protein names (first 5): {binarized_clean.columns[:5].tolist()}')
print(f'Protein names (last 5): {binarized_clean.columns[-5:].tolist()}')

Isotype control columns (9):
  mouseIgG1-kappaisotypeCtrl
  mouseIgG2a-kappaisotypeCtrl
  mouseIgG2b-kappaisotypeCtrl
  ArmenianHamsterIgGIsotypeCtrl
  ratIgG2b-kappaIsotypeCtrl
  ratIgG1-kappaisotypeCtrl
  ratIgG2a-kappaIsotypeCtrl
  ratIgG2c-kappaIsotypeCtrl
  ratIgG1-lambdaIsotypeCtrl

After dropping isotype controls: (180794, 257)
Protein names (first 5): ['C5L2', 'Cadherin11', 'CCR10-1', 'CD101-BB27', 'CD102-ICAM-2']
Protein names (last 5): ['CD98', 'HLA-A-B-C', 'HLA-DR', 'IgD', 'IgGFc']


## Step 4 — Create new MuData with binarized proteins

In [11]:
# Reindex binarized data to match MuData cell order (if needed)
rna_adata = mdata.mod['rna']
binarized_aligned = binarized_clean.loc[rna_adata.obs_names]
print(f'Aligned binarized shape: {binarized_aligned.shape}')
print(f'Order matches RNA: {list(binarized_aligned.index) == list(rna_adata.obs_names)}')

Aligned binarized shape: (180794, 257)
Order matches RNA: True


In [12]:
# Create new protein AnnData with binarized values
protein_adata = ad.AnnData(
    X=csr_matrix(binarized_aligned.values.astype(np.float32)),
    obs=rna_adata.obs.copy(),
    var=pd.DataFrame(index=binarized_aligned.columns)
)
print(f'New protein AnnData: {protein_adata.shape}')
print(f'Values are 0/1: min={protein_adata.X.min()}, max={protein_adata.X.max()}')

New protein AnnData: (180794, 257)
Values are 0/1: min=0.0, max=1.0


In [13]:
# Create new MuData
new_mdata = mu.MuData({'rna': rna_adata, 'protein': protein_adata})
print(f'New MuData: {new_mdata.shape}')
print(f'RNA: {new_mdata.mod["rna"].shape}')
print(f'Protein: {new_mdata.mod["protein"].shape}')

New MuData: (180794, 33951)
RNA: (180794, 33694)
Protein: (180794, 257)


/opt/anaconda3/envs/teo_citeprep_env/lib/python3.11/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/opt/anaconda3/envs/teo_citeprep_env/lib/python3.11/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


In [15]:
import anndata
output_path = DATA_PATH + 'GSE194315_binarized_mu.h5mu'
anndata.settings.allow_write_nullable_strings = True
new_mdata.write(output_path)
print(f'Saved to: {output_path}')

Saved to: /Users/teosecilmis/Documents/CITESEQ/Teo_datasets/Hackathon files/GSE194315_binarized_mu.h5mu


/opt/anaconda3/envs/teo_citeprep_env/lib/python3.11/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/opt/anaconda3/envs/teo_citeprep_env/lib/python3.11/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)
